# Fase 2.3 — EDA temporal: cosechas, supervivencia y deriva

**Tesis v2 · Eider Johan Díaz Berrío · MCDA EAFIT**

El notebook 05 mostró que la cartera cambió mucho entre 2022 y 2026. Aquí cuantifico ese cambio desde tres ángulos:

1. **Desempeño por cosecha (*vintage*)**: cómo se acumula la mora según los meses en libros, por cohorte.
2. **Supervivencia**: el tiempo hasta 90+ con censura, por grupos.
3. **Deriva de las variables**: el PSI por trimestre frente a un periodo de referencia, y la distancia entre FINTECH y Home Credit, que en la v1 aportaba el 80 % del peso del entrenamiento.

Las curvas de cosecha y de supervivencia describen el desempeño de la cartera e incluyen todas las cohortes, también 2025+. No se usan para elegir variables, así que no comprometen el *backtest* secundario.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
from statsmodels.duration.survfunc import survdiff

from tesis_v2 import config, estadistica as est, figuras, io_datos

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170, "display.max_columns", 30, "display.max_rows", 60)
figuras.estilo()

ds = io_datos.leer_parquet(config.PRIVADO / "dataset_analitico_v2.parquet")
lib = ds[(ds["producto"] == "libranza") & (ds["anio_cohorte"] != 2021)].reset_index(drop=True)
CORTE = pd.Timestamp("2026-10-08")
print(f"Créditos de libranza (2022+): {len(lib)}")

Créditos de libranza (2022+): 1083


## 1. Curvas de cosecha

Para cada cohorte anual y cada mes en libros (MOB) m, calculo el porcentaje de créditos que **alguna vez** alcanzó 30+ o 90+ días de mora hasta el mes m. Solo cuento los créditos cuya cohorte ya cumplió m meses al corte. Un crédito que cerró antes conserva su estado final ("*ever-bad*"). Es la vista clásica de gestión de cartera y no depende de la definición de elegibilidad.

In [2]:
mob = np.arange(1, 25)
filas = []
for coh, g in lib.groupby("anio_cohorte"):
    meses_disp = (CORTE - g["t_ref"]).dt.days / 30.44
    for m in mob:
        expuestos = g[meses_disp >= m]
        if len(expuestos) < 20:
            continue
        for umbral in (30, 90):
            malo = expuestos[f"dias_hasta_{umbral}"] <= m * 30.44
            filas.append({"cohorte": coh, "mob": m, "umbral": umbral, "creditos": len(expuestos),
                          "pct_ever": 100 * malo.mean()})
vint = pd.DataFrame(filas)
io_datos.guardar_resultado(vint.round(3), "fase2", "curvas_cosecha")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharex=True)
for ax, umbral in zip(axes, (30, 90)):
    for (coh, g), col in zip(vint[vint["umbral"] == umbral].groupby("cohorte"), figuras.SECUENCIA):
        ax.plot(g["mob"], g["pct_ever"], color=col, lw=1.7, label=f"{coh} (n={g['creditos'].iloc[0]})")
    ax.set_title(f"Alguna vez en mora {umbral}+ días")
    ax.set_xlabel("Meses en libros")
    ax.set_ylabel("% de la cohorte")
axes[0].legend(title="Cohorte", fontsize=7.5)
fig.suptitle("Curvas de cosecha (vintage) — libranza", fontweight="bold", fontsize=10)
fig.tight_layout()
figuras.guardar(fig, "fase2", "E09", "curvas_cosecha",
                "La cohorte 2022 es atípica (11,5 % alguna vez 90+ al mes 12); 2023–2025 son estables en 90+ (2,4–2,9 %), aunque 2024 tuvo más mora temprana 30+.")

**Lectura.** Las curvas permiten leer la madurez de cada cohorte sin esperar los 12 meses.

- La cohorte 2022, la de arranque con 52 créditos, es atípica: el 11,5 % llegó alguna vez a 90+ al mes 12 y el 23 % a 30+.
- De 2023 a 2025 la mora 90+ al mes 12 es muy estable (2,4 %–2,9 %).
- La cohorte 2024 tuvo bastante más mora temprana (14,5 % a 30+ al mes 12, frente a 9 % en 2023 y 2025), pero casi toda se curó antes de llegar a 90+.
- La mora 90+ no aparece antes del cuarto mes en libros, como se espera mecánicamente: hay que dejar de pagar tres cuotas.

Esto confirma que 30+ y 90+ miden fenómenos distintos (atraso temporal frente a incumplimiento) y que el objetivo 30+/6m es un análisis de sensibilidad, no un sustituto.

## 2. Supervivencia por grupos

Uso el objetivo de supervivencia construido en la Fase 1: el tiempo hasta 90+, censurado en el fin de observación. Comparo grupos con Kaplan-Meier y la prueba de *log-rank*.

In [3]:
def km_por_grupo(var, etiquetas, ax, titulo):
    filas = []
    for (val, g), col in zip(lib.groupby(var), figuras.SECUENCIA):
        km = est.kaplan_meier(g["tiempo_90"], g["evento_90"], grilla=np.arange(0, 731, 15))
        ax.step(km["t"] / 30.44, 100 * km["incidencia"], where="post", color=col, lw=1.6,
                label=f"{etiquetas.get(val, val)} (n={len(g)}, eventos={int(g['evento_90'].sum())})")
        k12 = est.kaplan_meier(g["tiempo_90"], g["evento_90"], grilla=[365.25]).iloc[0]
        filas.append({"variable": var, "grupo": etiquetas.get(val, val), "creditos": len(g),
                      "eventos": int(g["evento_90"].sum()), "incidencia_12m_%": round(100 * k12["incidencia"], 2),
                      "ic95_%": f"{100 * k12['inc_inf']:.1f}–{100 * k12['inc_sup']:.1f}"})
    chi2, p = survdiff(lib["tiempo_90"], lib["evento_90"], lib[var])
    ax.set_title(f"{titulo}\nlog-rank: χ² = {chi2:.1f}, p = {p:.3f}", fontsize=9)
    ax.set_xlabel("Meses desde el desembolso")
    ax.set_xlim(0, 24)
    ax.legend(fontsize=7)
    return pd.DataFrame(filas).assign(logrank_p=round(p, 4))


fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
r1 = km_por_grupo("anio_cohorte", {}, axes[0], "Por cohorte")
r2 = km_por_grupo("es_recurrente", {0: "Nuevo", 1: "Recurrente"}, axes[1], "Cliente nuevo o recurrente")
lib["monto_terciles"] = pd.qcut(lib["monto"], 3, labels=["Monto bajo", "Monto medio", "Monto alto"])
r3 = km_por_grupo("monto_terciles", {}, axes[2], "Por terciles de monto")
axes[0].set_ylabel("Incidencia acumulada de 90+ (%)")
fig.suptitle("Tiempo hasta 90+ días de mora — Kaplan-Meier", fontweight="bold", fontsize=10)
fig.tight_layout()
km_tab = pd.concat([r1, r2, r3], ignore_index=True)
io_datos.guardar_resultado(km_tab, "fase2", "km_por_grupo")
figuras.guardar(fig, "fase2", "E10", "km_por_grupo",
                "Solo la cohorte separa las curvas (log-rank p = 0,016, por el arranque 2022); recurrencia y monto apuntan en la dirección esperada pero no son significativos.")
km_tab

,variable,grupo,creditos,eventos,incidencia_12m_%,ic95_%,logrank_p
0,anio_cohorte,2022,52,8,12.69,5.9–26.2,0.0161
1,anio_cohorte,2023,208,6,7.11,3.0–16.5,0.0161
2,anio_cohorte,2024,275,8,4.97,2.3–10.8,0.0161
3,anio_cohorte,2025,226,7,3.74,1.7–8.2,0.0161
4,anio_cohorte,2026,322,0,0.00,0.0–0.0,0.0161
5,es_recurrente,Nuevo,572,16,7.44,4.3–12.7,0.3714
6,es_recurrente,Recurrente,511,13,4.15,2.2–7.9,0.3714
7,monto_terciles,Monto bajo,361,5,2.82,0.9–9.0,0.7145
8,monto_terciles,Monto medio,361,8,5.19,2.5–10.4,0.7145
9,monto_terciles,Monto alto,361,16,7.17,4.1–12.4,0.7145


**Lectura.** El análisis de supervivencia aprovecha más información que el bivariado: incluye créditos inmaduros y no descarta los cierres tempranos.

- **Cohorte:** es la única separación estadísticamente distinguible (log-rank p = 0,016). La incidencia a 12 meses baja de 12,7 % en 2022 a 7,1 % en 2023, 5,0 % en 2024 y 3,7 % en 2025. La cohorte 2026 todavía no tiene eventos porque casi ningún crédito ha cumplido cuatro meses. Parte de la caída puede ser real (la fintech aprendió a originar) y parte un efecto de la política de aprobación, que se volvió más estricta (Notebook 05).
- **Recurrencia:** los clientes nuevos tienen más incidencia que los recurrentes (7,4 % frente a 4,2 %), pero p = 0,37.
- **Monto:** la incidencia crece con el monto (2,8 % → 5,2 % → 7,2 %), coherente con el IV del Notebook 06, pero p = 0,71.

Ninguna de estas comparaciones ajusta por las demás: son descriptivas. La caída por cohorte es la principal razón para que el protocolo trate el tiempo explícitamente: la validación agrupada lo mezcla y el *backtest* lo aísla.

## 3. Tasa de evento por semestre de originación

Para ver si el riesgo de la cartera cambió en el tiempo, reporto la tasa 90+/12m (población elegible) y 30+/6m por semestre, con su IC de Wilson.

In [4]:
lib["semestre"] = lib["t0"].dt.year.astype(str) + "-S" + np.where(lib["t0"].dt.month <= 6, "1", "2")
el90 = lib[lib["elegible"]]
el30 = lib[lib["madura_6m"] & (lib["cierre_v2"] != "INDETERMINADO")]
t90 = est.tabla_tasa(el90, "semestre", "y_90_12m").assign(objetivo="90+/12m")
t30 = est.tabla_tasa(el30, "semestre", "y_30_6m").assign(objetivo="30+/6m")
sem = pd.concat([t90, t30])
io_datos.guardar_resultado(sem, "fase2", "tasa_por_semestre")

fig, ax = plt.subplots(figsize=(8, 3.6))
for (obj, g), col, dx in zip(sem.groupby("objetivo", sort=False), [figuras.AZUL, figuras.NARANJA], [-0.12, 0.12]):
    x = np.arange(len(g)) + dx
    ax.errorbar(x, g["tasa_%"], yerr=[g["tasa_%"] - g["ic95_inf_%"], g["ic95_sup_%"] - g["tasa_%"]], fmt="o-",
                color=col, ecolor=col, alpha=0.9, capsize=2, ms=4, lw=1, label=obj)
ax.set_xticks(np.arange(len(t90)), t90["semestre"], rotation=45)
ax.set_ylabel("Tasa de evento (%) e IC 95 %")
ax.set_title("Tasa de incumplimiento por semestre de originación — libranza")
ax.legend()
figuras.guardar(fig, "fase2", "E11", "tasa_por_semestre",
                "Fuera del arranque de 2022, la tasa 90+/12m oscila entre 1 % y 4 % por semestre; los IC se superponen y no se distingue deriva del evento.")
sem[["objetivo", "semestre", "creditos", "eventos", "tasa_%", "ic95_inf_%", "ic95_sup_%"]]

,objetivo,semestre,creditos,eventos,tasa_%,ic95_inf_%,ic95_sup_%
0,90+/12m,2022-S1,13,1,7.69,1.37,33.31
1,90+/12m,2022-S2,34,3,8.82,3.05,22.96
2,90+/12m,2023-S1,92,1,1.09,0.19,5.90
3,90+/12m,2023-S2,105,4,3.81,1.49,9.39
4,90+/12m,2024-S1,110,2,1.82,0.50,6.39
5,90+/12m,2024-S2,140,5,3.57,1.53,8.09
6,90+/12m,2025-S1,106,4,3.77,1.48,9.30
7,90+/12m,2025-S2,99,1,1.01,0.18,5.50
8,90+/12m,2026-S1,31,0,0.00,0.00,11.03
9,90+/12m,2026-S2,1,0,0.00,0.00,79.35


## 4. Deriva de los predictores: PSI por trimestre

El PSI (*Population Stability Index*) compara la distribución de una variable en un periodo con la de un periodo de referencia. Uso como referencia el año 2023, el primero con volumen suficiente y completo dentro del desarrollo. Los umbrales convencionales son: < 0,10 estable, 0,10–0,25 cambio moderado, > 0,25 cambio importante.

In [5]:
VARS_PSI = ["monto", "plazo_meses", "tasa_mensual", "edad_t0", "n_creditos_previos", "peor_mora_previa",
            "meses_desde_primer_credito", "emp_creditos_previos", "recaudo", "tipo_solicitud", "cierre_previo",
            "tiene_aval", "cobra_seguro", "cobra_estudio", "cobra_4x1000"]
ref = lib[lib["anio_cohorte"] == 2023]
trims = sorted(lib["trimestre_t0"].unique())
trims = [q for q in trims if (lib["trimestre_t0"] == q).sum() >= 20]
psi = pd.DataFrame({q: {v: est.psi(ref[v], lib.loc[lib["trimestre_t0"] == q, v]) for v in VARS_PSI} for q in trims})
io_datos.guardar_resultado(psi.round(3).reset_index(names="variable"), "fase2", "psi_por_trimestre")

fig, ax = plt.subplots(figsize=(10, 0.36 * len(VARS_PSI) + 1.5))
from matplotlib.colors import BoundaryNorm, ListedColormap
cmap = ListedColormap(["#e8f1e4", "#fbe3c4", "#f2a9a0"])
norm = BoundaryNorm([0, 0.10, 0.25, 50], 3)
ax.imshow(psi.values, aspect="auto", cmap=cmap, norm=norm)
for i in range(psi.shape[0]):
    for j in range(psi.shape[1]):
        val = psi.values[i, j]
        ax.text(j, i, f"{val:.2f}" if val < 10 else ">10", ha="center", va="center", fontsize=6.3)
ax.set_xticks(range(len(trims)), trims, rotation=60, fontsize=7)
ax.set_yticks(range(len(VARS_PSI)), VARS_PSI, fontsize=7.5)
ax.grid(False)
ax.set_title("PSI por trimestre frente a 2023 (verde < 0,10; naranja 0,10–0,25; rojo > 0,25)")
figuras.guardar(fig, "fase2", "E12", "psi_por_trimestre",
                "Recaudo, aval, tasa, tipo de solicitud y cobros cambian de forma abrupta (política); la historia del cliente y del empleador deriva por maduración; la edad es la más estable.")

resumen_psi = pd.DataFrame({"psi_mediano": psi.median(axis=1), "psi_maximo": psi.max(axis=1),
                            "trimestres_>0.25": (psi > 0.25).sum(axis=1)}).round(3).sort_values("psi_mediano", ascending=False)
io_datos.guardar_resultado(resumen_psi.reset_index(names="variable"), "fase2", "psi_resumen")
resumen_psi

,psi_mediano,psi_maximo,trimestres_>0.25
recaudo,7.694,7.694,16
tiene_aval,7.045,18.421,9
tasa_mensual,4.945,7.833,14
emp_creditos_previos,3.390,5.904,16
tipo_solicitud,3.052,9.202,14
cobra_4x1000,2.678,7.394,10
meses_desde_primer_credito,1.481,2.623,16
n_creditos_previos,0.981,1.736,16
monto,0.440,1.459,10
cierre_previo,0.437,1.690,13


**Lectura.** El mapa separa con nitidez dos tipos de variables:

- **Variables de política o de registro** (recaudo, aval, tasa mensual, tipo de solicitud, seguro, estudio, 4x1000): saltan de golpe, con PSI muy por encima de 0,25 durante trimestres enteros (medianas entre 2,7 y 7,7). No describen al cliente sino decisiones de la fintech en cada época. Un modelo que las use aprende la época, y en producción se desajusta con cada cambio de política. La tasa mensual es precio fijado por política y, además, probablemente depende del riesgo percibido: no debe entrar como predictor.
- **Variables del crédito** (monto, plazo): el plazo cambia moderadamente (PSI mediano 0,27). El monto cambia bastante (0,44), coherente con montos más altos en el tiempo. Esto pesa sobre la única variable con IV por encima del ruido, porque parte de su señal puede ser temporal.
- **Edad**: es la más estable (PSI mediano 0,20; supera 0,25 solo en 5 trimestres).
- La historia del cliente (créditos previos, antigüedad) y la del empleador derivan **por construcción**: crecen a medida que la cartera madura. No es un problema de calidad, pero obliga a que el modelo se recalibre periódicamente. Es parte del protocolo de monitoreo de la Fase 6.

## 5. ¿Qué tan parecida es FINTECH a Home Credit?

En la v1, Home Credit aportaba el 80 % del peso del entrenamiento. Si sus distribuciones son muy distintas de las de FINTECH, las pendientes aprendidas allí no se trasladan bien. Comparo las variables que tienen una definición comparable en ambas fuentes (las monetarias están en monedas distintas y quedan fuera):

- edad en t0, plazo en meses, número de créditos previos, peor mora previa y antigüedad como cliente;
- con tres medidas: diferencia de medias estandarizada (SMD), estadístico de Kolmogorov-Smirnov y PSI (referencia: Home Credit);
- y un **clasificador adversarial**, un *gradient boosting* que intenta adivinar de qué fuente viene cada registro. Un AUC cercano a 0,5 indica fuentes intercambiables; cercano a 1, fuentes separables.

Home Credit es un conjunto público y no tiene datos personales identificables.

In [6]:
hc = io_datos.leer_parquet(config.HOME_CREDIT_MAPEADO, ["edad_t0", "plazo_meses", "n_creditos_previos",
                                                        "peor_mora_previa", "antiguedad_cliente_meses", "y"])
hc = hc.rename(columns={"antiguedad_cliente_meses": "meses_desde_primer_credito"})
fin = lib.copy()
fin_cmp = fin[["edad_t0", "plazo_meses", "n_creditos_previos", "peor_mora_previa", "meses_desde_primer_credito"]].astype(float)
CMP = list(fin_cmp.columns)
filas = []
for v in CMP:
    a, b = fin_cmp[v].dropna(), hc[v].dropna().astype(float)
    smd = (a.mean() - b.mean()) / np.sqrt((a.var() + b.var()) / 2)
    filas.append({"variable": v, "media_fintech": a.mean(), "media_home_credit": b.mean(), "mediana_fintech": a.median(),
                  "mediana_home_credit": b.median(), "SMD": smd, "KS": stats.ks_2samp(a, b).statistic,
                  "PSI": est.psi(hc[v].astype(float), fin_cmp[v]), "%_faltante_fintech": 100 * fin_cmp[v].isna().mean(),
                  "%_faltante_home_credit": 100 * hc[v].isna().mean()})
cmp_tab = pd.DataFrame(filas).round(3)
io_datos.guardar_resultado(cmp_tab, "fase2", "fintech_vs_home_credit")
print(f"Tasa de evento: Home Credit {100 * hc['y'].mean():.1f} % (dificultad de pago) | FINTECH libranza elegible "
      f"{100 * lib.loc[lib['elegible'], 'y_90_12m'].mean():.1f} % (90+/12m)")
cmp_tab

Tasa de evento: Home Credit 9.0 % (dificultad de pago) | FINTECH libranza elegible 2.9 % (90+/12m)


,variable,media_fintech,media_home_credit,mediana_fintech,mediana_home_credit,SMD,KS,PSI,%_faltante_fintech,%_faltante_home_credit
0,edad_t0,38.816,40.658,36.936,40.104,-0.176,0.119,0.079,0.277,0.000
1,plazo_meses,18.109,21.720,13.436,20.561,-0.311,0.320,0.599,0.000,0.000
2,n_creditos_previos,1.010,2.900,0.000,2.000,-1.034,0.476,0.936,0.000,0.000
3,peor_mora_previa,15.773,18.521,7.000,1.000,-0.034,0.259,1.628,52.816,5.412
4,meses_desde_primer_credito,17.084,51.131,13.765,50.000,-1.582,0.540,4.250,52.816,5.252


In [7]:
rng = np.random.default_rng(config.SEMILLA)
hc_m = hc.iloc[rng.choice(len(hc), size=10 * len(fin_cmp), replace=False)][CMP].astype(float)
X = pd.concat([fin_cmp, hc_m], ignore_index=True)
y = np.r_[np.ones(len(fin_cmp)), np.zeros(len(hc_m))]
clf = HistGradientBoostingClassifier(max_depth=3, max_iter=200, learning_rate=0.05, random_state=config.SEMILLA)
p = cross_val_predict(clf, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=config.SEMILLA), method="predict_proba")[:, 1]
auc_adv = roc_auc_score(y, p)
por_var = {v: roc_auc_score(y, cross_val_predict(clf, X[[v]], y, cv=StratifiedKFold(5, shuffle=True, random_state=config.SEMILLA),
                                                 method="predict_proba")[:, 1]) for v in CMP}
adv = pd.DataFrame({"variables": ["todas"] + CMP, "auc_adversarial": [auc_adv] + list(por_var.values())}).round(3)
io_datos.guardar_resultado(adv, "fase2", "clasificador_adversarial")
adv

,variables,auc_adversarial
0,todas,0.963
1,edad_t0,0.569
2,plazo_meses,0.861
3,n_creditos_previos,0.792
4,peor_mora_previa,0.817
5,meses_desde_primer_credito,0.891


In [8]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, v, lim in zip(axes, ["edad_t0", "plazo_meses", "n_creditos_previos", "meses_desde_primer_credito"],
                      [(18, 75), (0, 60), (0, 10), (0, 100)]):
    bins = np.linspace(*lim, 30)
    ax.hist(hc[v].dropna().clip(*lim), bins=bins, density=True, color=figuras.GRIS, alpha=0.55, label="Home Credit")
    ax.hist(fin_cmp[v].dropna().clip(*lim), bins=bins, density=True, histtype="step", color=figuras.AZUL, lw=1.8,
            label="FINTECH libranza")
    ax.set_title(v, fontsize=8.5)
    ax.set_yticks([])
axes[0].legend(fontsize=7)
fig.suptitle(f"FINTECH frente a Home Credit — AUC del clasificador adversarial = {auc_adv:.2f}", fontweight="bold", fontsize=10)
fig.tight_layout()
figuras.guardar(fig, "fase2", "E13", "fintech_vs_home_credit",
                "Un clasificador distingue casi perfectamente la fuente de cada registro: Home Credit no es intercambiable con la cartera de libranza.")

[Figura retirada del repositorio público: su título nombraba a la entidad.]


**Lectura.** Las dos fuentes son claramente distintas. El clasificador adversarial separa los registros con un **AUC de 0,96**. Solo la edad es parecida (SMD −0,18, AUC individual 0,57). El plazo, los créditos previos, la peor mora previa y la antigüedad como cliente tienen AUC individuales de 0,79 a 0,89. La antigüedad en Home Credit triplica la de FINTECH (mediana de 50 frente a 14 meses), y la peor mora previa tiene casi un 53 % de faltantes estructurales en FINTECH (clientes nuevos) frente a un 5 % en Home Credit. A esto se suma que el evento no es el mismo (dificultad de pago frente a 90+/12m) y que la tasa base triplica la de FINTECH. La conclusión metodológica respalda la decisión 3 del Project: Home Credit **no** debe entrar como datos apilados con un peso fijo, porque el modelo aprendería sobre todo la geografía de otra población. Su papel defendible es el de **información previa** sobre la dirección y la magnitud aproximada de efectos generales (edad, plazo, mora previa), con una escala que los datos propios puedan corregir. Esa escala la fija el protocolo (M3, parámetro τ).

In [9]:
figuras.escribir_catalogo("fase2", "Figuras de la Fase 2 (EDA)")
_ = io_datos.escribir_registro("07_eda_temporal_y_deriva", {
    "creditos": len(lib), "auc_adversarial": round(float(auc_adv), 3), "referencia_psi": "cohorte 2023",
    "figuras": ["E09", "E10", "E11", "E12", "E13"]})

## Conclusiones

- La cohorte de arranque (2022) es atípica. De 2023 a 2025 la mora 90+ al mes 12 es estable (2,4–2,9 %), aunque 2024 tuvo más mora temprana que se curó. La mora 90+ no aparece antes del cuarto mes en libros.
- La cohorte es la única separación significativa en supervivencia (log-rank p = 0,016). Recurrencia y monto apuntan en la dirección esperada sin ser significativos.
- Por semestre, fuera de 2022, la tasa 90+/12m no muestra una deriva distinguible.
- Las variables de **política y registro** (recaudo, aval, tasa, tipo de solicitud, cobros) cambian de forma abrupta en el tiempo. Deben quedar fuera del núcleo del modelo. El monto también deriva (PSI mediano 0,44) y la edad es la más estable.
- **Home Credit es otra población**: un clasificador adversarial la separa de FINTECH con un AUC de 0,96. Esto respalda usarla como *prior* con escala ajustable y no como datos apilados.